## NN-Module

In [19]:
import torch
import torch.nn as nn

class Model(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear(features)
        out = self.sigmoid(out)

        return out

In [20]:
#create dataset

features = torch.rand(10,5)
model = Model(features.shape[1])

model(features)

tensor([[0.4610],
        [0.4997],
        [0.3351],
        [0.4080],
        [0.3647],
        [0.4317],
        [0.3760],
        [0.3261],
        [0.3762],
        [0.4127]], grad_fn=<SigmoidBackward0>)

In [21]:
model.linear.bias

Parameter containing:
tensor([0.0958], requires_grad=True)

## MultiLayer (hidden layer)

In [24]:
class Model(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.linear1 = nn.Linear(num_features, 3)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(3, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear1(features)
        out = self.relu(out)
        out = self.linear2(out)
        out = self.sigmoid(out)

        return out

In [25]:
#create dataset

features = torch.rand(10,5)
model = Model(features.shape[1])

model(features)

tensor([[0.4652],
        [0.4652],
        [0.4652],
        [0.4659],
        [0.4652],
        [0.4652],
        [0.4652],
        [0.4652],
        [0.4652],
        [0.4652]], grad_fn=<SigmoidBackward0>)

## Sequential Container

In [26]:
class Model(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 3) ,nn.ReLU()
        ,nn.Linear(3, 1)
        ,nn.Sigmoid()
        )

    def forward(self, features):
        out = self.network(features)

        return out

## Creating Pipeline

In [27]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [28]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

In [29]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [30]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [38]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:,0], test_size=0.2)

In [39]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

In [52]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [53]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor  = torch.from_numpy(X_test).float()

y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor  = torch.from_numpy(y_test).float()


In [57]:
class MySimpleNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features,1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, X):
        out = self.linear(X)
        out = self.sigmoid(out)
        return out

In [55]:
learning_rate= 0.1
epochs = 25

In [58]:
loss_function = nn.BCELoss()

In [71]:
#create model
model = MySimpleNN(X_train_tensor.shape[1])

optim = torch.optim.SGD(model.parameters(), lr=learning_rate)
#defineloop
for epoch in range(epochs):
    #forwardpass
    y_pred = model(X_train_tensor)
    loss = loss_function(y_pred, y_train_tensor.view(-1,1))
    
    optim.zero_grad()
    #backwardpass
    loss.backward()

    #parametersupdate
    optim.step()
    
    #zerogradients

    print (f'Epoch: {epoch + 1}, Loss: {loss.item()}')


Epoch: 1, Loss: 0.7060208916664124
Epoch: 2, Loss: 0.5429048538208008
Epoch: 3, Loss: 0.4559154212474823
Epoch: 4, Loss: 0.40155500173568726
Epoch: 5, Loss: 0.3635677993297577
Epoch: 6, Loss: 0.33506640791893005
Epoch: 7, Loss: 0.3126334846019745
Epoch: 8, Loss: 0.2943623661994934
Epoch: 9, Loss: 0.2790951728820801
Epoch: 10, Loss: 0.2660825848579407
Epoch: 11, Loss: 0.2548151910305023
Epoch: 12, Loss: 0.24493296444416046
Epoch: 13, Loss: 0.23617298901081085
Epoch: 14, Loss: 0.22833825647830963
Epoch: 15, Loss: 0.22127769887447357
Epoch: 16, Loss: 0.21487319469451904
Epoch: 17, Loss: 0.20903082191944122
Epoch: 18, Loss: 0.2036748230457306
Epoch: 19, Loss: 0.19874334335327148
Epoch: 20, Loss: 0.19418512284755707
Epoch: 21, Loss: 0.1899574100971222
Epoch: 22, Loss: 0.18602408468723297
Epoch: 23, Loss: 0.18235445022583008
Epoch: 24, Loss: 0.17892210185527802
Epoch: 25, Loss: 0.1757042407989502


In [72]:
model.linear.weight

Parameter containing:
tensor([[ 0.1872,  0.3179,  0.2020,  0.2516,  0.0718, -0.0431,  0.0267,  0.1831,
          0.1573, -0.0245,  0.0424, -0.0664,  0.3435,  0.3271, -0.0468,  0.0119,
          0.0089,  0.2044, -0.1844, -0.0166,  0.2985,  0.0530,  0.2527,  0.2341,
          0.0961,  0.0764,  0.3021,  0.3606,  0.1741,  0.1788]],
       requires_grad=True)

In [73]:
model.linear.bias

Parameter containing:
tensor([-0.2185], requires_grad=True)

In [74]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.9).float()

print(y_pred)

tensor([[0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
      

In [75]:
accuracy = (y_pred == y_test_tensor).float().mean()
print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5689442753791809
